In [ ]:
import sys, os, warnings
from pathlib import Path

import numpy as np

NOTEBOOK_DIR = Path().resolve()                    # tests/
SRC_DIR      = NOTEBOOK_DIR.parent / "tsforge"         # src/

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Imports ───────────────────────────────────────────────────────────────────
from common.ensembling import Ensembler

warnings.filterwarnings("ignore")
print("imports OK")


In [ ]:
T, H, C = 10, 5, 2
pattern = np.array([1, 2, 3, 4, 5], dtype=float)
preds_single = np.stack([np.roll(pattern, -t) for t in range(T)])  # (T, H)

def make_preds(Q):
    """(1, T, H, C, Q) — same pattern broadcast across C and Q."""
    p = np.stack([preds_single, preds_single], axis=-1)[None]  # (1, T, H, C)
    return np.stack([p] * Q, axis=-1)                          # (1, T, H, C, Q)

In [ ]:
# ── TEST: multiple quantiles (Q=3), stride=1, identity ────────────────
Q = 3
preds = make_preds(Q)

ensembler = Ensembler(ensemble_method='identity', stride=1)
out = ensembler.ensemble(preds)

assert out.shape == (1, T, H, C, Q) 
assert np.allclose(out, preds, atol=1e-9)
print("TEST PASSED\n")

In [ ]:
# ── TEST: multiple quantiles (Q=3), stride=1, mean ───────────────────────────────
Q = 3
preds = make_preds(Q)

ensembler_q1 = Ensembler(ensemble_method='mean', stride=1)
ensembler_q3 = Ensembler(ensemble_method='mean', stride=1)

preds_q1 = make_preds(1)
out_q1 = ensembler_q1.ensemble(preds_q1)  # (1, T, H, C, 1)
out_q3 = ensembler_q3.ensemble(preds)     # (1, T, H, C, 3)

assert out_q3.shape == (1, T, H, C, Q)

for q in range(Q):
    assert np.allclose(out_q3[..., q], out_q1[..., 0], atol=1e-9)
print("TEST PASSED\n")


In [ ]:
# ── TEST: stride=2, identity round-trip (Q=1 and Q=3) ────────────────────────────────────
for Q in [1, 3]:
    preds = make_preds(Q)
    ensembler = Ensembler(ensemble_method='identity', stride=2)
    out = ensembler.ensemble(preds)
    assert out.shape == (1, T, H, C, Q)
    assert np.allclose(out, preds, atol=1e-9)
print("TEST PASSED\n")

In [ ]:
# ── TEST 4: stride=2, mean (Q=3) ───────────────
Q = 3
preds = make_preds(Q)
ensembler = Ensembler(ensemble_method='mean', stride=2)
out = ensembler.ensemble(preds)

assert out.shape == (1, T, H, C, Q)
# stride=2 with identity pattern: dates covered by >1 window should be averaged
# verify each quantile slice is identical (same pattern across Q)
for q in range(1, Q):
    assert np.allclose(out[..., q], out[..., 0], atol=1e-9)
print("TEST PASSED\n")



In [ ]:
# ── TEST: stride=H raises ValueError ───────────────────────────────────
preds = make_preds(1)
ensembler = Ensembler(ensemble_method='mean', stride=H)
try:
    ensembler.ensemble(preds)
except ValueError as e:
    print(e)
    print("TEST PASSED\n")
else:
    raise AssertionError("expected a ValueError for stride=H, none was raised")

In [ ]:
# ── TEST: stride=H + 1 raises ValueError ───────────────────────────────────
preds = make_preds(1)
ensembler = Ensembler(ensemble_method='mean', stride=H + 1)
try:
    ensembler.ensemble(preds)
except ValueError as e:
    print(e)
    print("TEST PASSED\n")
else:
    raise AssertionError("expected a ValueError for stride=H + 1, none was raised")

In [ ]:
# ── TEST: ensembling is CAUSAL — window t may only pool windows 0..t ──────────
# Tag every prediction with its (FCD, lead) as 10*t + h. With stride=1 the target
# date is t + h, so we can check the pooled set exactly.
n, hz = 5, 5
tagged = np.zeros((1, n, hz, 1, 1))
for t in range(n):
    for h in range(hz):
        tagged[0, t, h, 0, 0] = 10 * t + h

out = Ensembler(ensemble_method='mean', stride=1).ensemble(tagged)

for t in range(n):
    for h in range(hz):
        date = t + h
        expected = [10 * p + (date - p) for p in range(t + 1) if 0 <= date - p < hz]
        assert np.isclose(out[0, t, h, 0, 0], np.mean(expected)), (
            f"window {t}, lead {h} pooled the wrong set"
        )

# the oldest window has no earlier forecast, so it must come back unchanged
assert np.allclose(out[0, 0], tagged[0, 0])
# changing only the last window must not disturb any earlier one
bumped = tagged.copy(); bumped[0, -1] += 1000.0
assert np.allclose(out[0, :-1], Ensembler(ensemble_method='mean', stride=1).ensemble(bumped)[0, :-1])
print("TEST PASSED\n")